In [1]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("API Key Loaded")

API Key Loaded


In [2]:
llm = ChatGroq(model = "openai/gpt-oss-120b")

### **Pydantic LLM Schema**

In [3]:
from pydantic import BaseModel, Field
from typing import List, TypedDict

class llm_schema(BaseModel):
    tasks:List[str] = Field(...,description = "A List Of Tasks To Be Performed By The Worker")
    
llm_with_schema = llm.with_structured_output(llm_schema)

### **Graph Schema**

In [4]:
class graph_schema(TypedDict):
    tasks:List[str]
    query:str
    result:List[str]
    summary:str

### **Creating Orchestrator Node**

In [5]:
def orchestrator_node(state:graph_schema) -> graph_schema:
    user_query = state['query']
    
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system","you are a orchestrator that breaks down a user query into tasks for the worker."),
            ("user",f"User Query: {user_query}. Please generate one prompt for the worker to complete.")
        ]
    )
    
    chain = prompt | llm_with_schema
    
    response = chain.invoke({"query":user_query})
    
    state['tasks'] = response.tasks
    
    return state

### **Worker Node**

In [6]:
def execute(query:str) :
    
    response = llm.invoke(f"Please Execute This task {query} ")
    
    return response

In [7]:
from concurrent.futures import ThreadPoolExecutor

def worker_node(state:graph_schema) -> graph_schema:
    
    tasks = state['tasks']
    results = []
    
    with ThreadPoolExecutor(max_workers= len(tasks)):
        results_futures = executer.map(execute, tasks)
        for result in results_futures:
                   results.append(result)
    
    state['results'] = results
    
    return state

### **Collector Node**